In [102]:
import aegnn
import torch

# Data visualization

In [103]:
import os
os.environ["AEGNN_DATA_DIR"] = "/home/kevin-imagine/Documents/event_graph/data"
print(os.environ["AEGNN_DATA_DIR"])

/home/kevin-imagine/Documents/event_graph/data


In [104]:
data_module = aegnn.datasets.NCars(batch_size=1, shuffle=False, )


In [105]:
data_module

NCars[Train: None
Validation: None]

In [106]:
data_module.setup()

In [107]:
data_module.train_dataset.files

['/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_7264',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_97',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_5024',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_1512',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_6704',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_1037',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_6195',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_9463',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_4835',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_11671',
 '/home/kevin-imagine/Documents/event_graph/data/ncars/processed/training/sequence_384',
 '/home/kevi

In [108]:
import glob
import os


In [109]:
data_module.root 

'/home/kevin-imagine/Documents/event_graph/data/ncars'

In [110]:
g = data_module.train_dataset[0]

In [111]:
g

Data(x=[10000, 1], pos=[10000, 3], file_id='sequence_7264', label=[1], y=[1], edge_index=[2, 315712], edge_attr=[315712, 3])

In [112]:
g.x.shape

torch.Size([10000, 1])

In [113]:
g.edge_index

tensor([[4906, 4584, 7533,  ...,  497, 8873, 4374],
        [   0,    0,    0,  ..., 9999, 9999, 9999]])

In [114]:
g.edge_attr

tensor([[0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000],
        ...,
        [0.6667, 0.5000, 0.5000],
        [0.6667, 0.5000, 0.5000],
        [0.6667, 0.5000, 0.5000]])

In [115]:
g.label

['background']

In [116]:
g.y

tensor([1])

# Data pre-processing

 Convert event stream in event graph

In [117]:
from aegnn.datasets.ncars import NCars

In [118]:
ncars = NCars()

In [119]:
ncars

NCars[Train: None
Validation: None]

In [120]:
event_sample_file = ncars.raw_files("training")[0]
event_sample_file

je vaius chercher ici: /home/kevin-imagine/Documents/event_graph/data/ncars/training/*


'/home/kevin-imagine/Documents/event_graph/data/ncars/training/sequence_7264'

In [121]:
ncars.classes

['car', 'background']

In [122]:
ncars.pre_transform

<bound method NCars.pre_transform of NCars[Train: None
Validation: None]>

In [123]:
#Loading raw event data from file name
event_sample = ncars.load(event_sample_file)
event_sample

Data(x=[10066, 1], pos=[10066, 3])

On a ici environ 10000 events dans le raw event cloud.
Chaque event est un noeud, ayant une feature (x) avec la polarité et une position spatiale (pos) avec les coordonnées (x,y,t) 

In [124]:
event_sample.label = ncars.read_label(event_sample_file)
event_sample

Data(x=[10066, 1], pos=[10066, 3], label='background')

In [125]:
class_dict = {class_id: i for i, class_id in enumerate(ncars.classes)}
event_sample.y = torch.tensor([class_dict[event_sample.label]])
event_sample


Data(x=[10066, 1], pos=[10066, 3], label='background', y=[1])

In [126]:
event_sample_data_pt = ncars.pre_transform(event_sample)
event_sample_data_pt

Data(x=[10000, 1], pos=[10000, 3], label='background', y=[1], edge_index=[2, 315654])

### Pre-transform

In [127]:
from aegnn.datasets.utils.normalization import normalize_time

In [128]:
event_sample.pos

tensor([[6.0000e+00, 3.1000e+01, 0.0000e+00],
        [2.9000e+01, 1.9000e+01, 1.0000e-11],
        [3.3000e+01, 1.7000e+01, 2.0000e-11],
        ...,
        [1.6000e+01, 2.1000e+01, 4.9966e-07],
        [3.1000e+01, 5.2000e+01, 4.9966e-07],
        [4.3000e+01, 2.4000e+01, 4.9967e-07]])

In [130]:
event_sample.pos[:, 2] = normalize_time(event_sample.pos[:, 2])
event_sample.pos

tensor([[6.0000e+00, 3.1000e+01, 0.0000e+00],
        [2.9000e+01, 1.9000e+01, 2.5000e-22],
        [3.3000e+01, 1.7000e+01, 5.0000e-22],
        ...,
        [1.6000e+01, 2.1000e+01, 1.2491e-17],
        [3.1000e+01, 5.2000e+01, 1.2492e-17],
        [4.3000e+01, 2.4000e+01, 1.2492e-17]])

La coordonnée temporelle a été normalisée 